# Kỹ thuật 1: CLAHE (Contrast Limited Adaptive Histogram Equalization)

Notebook này thực hiện các thí nghiệm minh họa cho Kỹ thuật 1 theo đúng yêu cầu Mục 2.6 trong đề tài.

### Có dùng AI hỗ trợ

* **Công cụ**: Claude 3.5 Sonnet / Gemini 2.5 Pro
* **Mục đích**: Thiết lập cấu hình hiển thị và import hàm tự cài đặt từ `src.clahe`.
* **Prompt**:
> Hãy import các thư viện NumPy, Matplotlib và các hàm tự viết trong `src.clahe` để đọc ảnh thử nghiệm từ `data/ex1.png`.
* **Đã sửa lại những gì**: Điều chỉnh đường dẫn thư mục `../` để có thể import chính xác từ thư mục `src`.

In [1]:
import sys
import os
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from src.clahe import clip_histogram, build_tile_luts, clahe

img = plt.imread('../data/ex1.jpg')

if img.ndim == 3:
    img = (img[:, :, 0] * 255).astype(np.uint8)
else:
    img = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img.astype(np.uint8)

plt.figure(figsize=(5, 5))
plt.imshow(img, cmap='gray')
plt.title('Anh goc (Anh sang khong deu + Nhieu)')
plt.axis('off')
plt.show()

C:\Users\hoang\AppData\Local\Temp\ipykernel_175476\1167145328.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## (a) Ảnh hưởng của tham số: Quét clip_limit và grid

### Có dùng AI hỗ trợ

* **Công cụ**: Claude 3.5 Sonnet
- **Muc dich**: Dung luoi anh 4x4 quet 4 gia tri grid va 4 gia tri clip_limit.
- **Prompt**:
> Viet doan code quet grid in [(2,2), (4,4), (8,8), (16,16)] va clip_limit in [1.0, 2.0, 5.0, 20.0] dung ham clahe tu viet va hien thi duoi dang ma tran anh 4x4 bang Matplotlib.
- **Da sua lai nhung gi**: Can chinh tieu de va layout hien thi.
## (b) So sanh voi thuat toan da hoc

### Co dung AI ho tro
- **Cong cu**: Gemini 2.5 Pro
- **Muc dich**: Cai dat doi sanh 4 phuong phap: Keo gian tuyen tinh, Gamma correction, HE toan cuc va CLAHE kem bieu do Histogram.
- **Prompt**:
> Viet code so sanh 4 ky thuat tang cuong tuong phan tren cung 1 anh: Linear stretching, Gamma (0.5), Global Histogram Equalization, va CLAHE; hien thi anh ket qua va bieu do histogram cua tung anh.
- **Da sua lai nhung gi**: Viet truc tiep ham tinh HE toan cuc bang CDF chuan.



In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# A) ANH HUONG CUA THAM SO
os.makedirs("result", exist_ok=True)

grids = [(2, 2), (4, 4), (8, 8), (16, 16)]
clip_limits = [1.0, 2.0, 5.0, 20.0]

fig1, axes1 = plt.subplots(4, 4, figsize=(14, 14))

for i, g in enumerate(grids):
    for j, cl in enumerate(clip_limits):
        # Đã sửa s_max=s thành clip_limit=cl
        out = clahe(img, grid=g, clip_limit=cl)

        name = f"Grid_{g[0]}x{g[1]}_Clip_{cl}.png"
        cv2.imwrite(f"result/{name}", out)

        axes1[i, j].imshow(out, cmap="gray")
        axes1[i, j].set_title(f"Grid: {g}\nClip: {cl}")
        axes1[i, j].axis("off")

plt.tight_layout()
plt.savefig("result/Grid_Clip_Results.png", dpi=200)
plt.show() 


# B) SO SANH THUAT TOAN DA HOC
# Keo gian tuong phan tuyen tinh
min_val, max_val = np.min(img), np.max(img)
linear_stretched = ((img - min_val) / (max_val - min_val) * 255.0).astype(np.uint8)

# Gamma correction (gamma = 0.5)
gamma = 0.5
gamma_corrected = np.clip(255.0 * (img / 255.0) ** gamma, 0, 255).astype(np.uint8)

# Can bang luoc do xam toan cuc (Global HE)
hist = np.bincount(img.ravel(), minlength=256)
cdf = np.cumsum(hist)
lut_he = np.round(255.0 * cdf / cdf[-1]).astype(np.uint8)
global_he = lut_he[img]

# CLAHE tu viet
# Đã sửa s_max=2.0 thành clip_limit=2.0
res_clahe = clahe(img, grid=(8, 8), clip_limit=2.0)

methods = [
    ('Keo gian tuyen tinh', linear_stretched),
    ('Hieu chinh Gamma (0.5)', gamma_corrected),
    ('Can bang HE toan cuc', global_he),
    ('CLAHE (8x8, clip=2.0)', res_clahe) # Đã sửa title
]

fig2, axes2 = plt.subplots(2, 4, figsize=(18, 8))
for col, (name, res) in enumerate(methods):
    axes2[0, col].imshow(res, cmap='gray')
    axes2[0, col].set_title(name)
    axes2[0, col].axis('off')
    
    axes2[1, col].hist(res.ravel(), bins=256, range=[0, 256], color='black', alpha=0.7)
    axes2[1, col].set_title(f"Histogram: {name}")
    axes2[1, col].set_xlim([0, 256])

plt.tight_layout()
plt.savefig("result/Compare_Methods.png", dpi=200)
plt.show()

C:\Users\hoang\AppData\Local\Temp\ipykernel_175476\2397630074.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show() # Thêm show() để hiển thị xong phần A
C:\Users\hoang\AppData\Local\Temp\ipykernel_175476\2397630074.py:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
